Note: to run this noteboook follow the instructions in the ```setup-old.sh``` file and activate the MolFLAE2 environment

In [1]:
import argparse
import torch 

from utils.config import load_config
from utils.data_loading import process_sdf_files_to_list
from model.encoder import Encoder

/home/teaching/miniconda3/envs/MolFLAE/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
parser = argparse.ArgumentParser()
parser.add_argument('--sdf_folder', type=str,default='data/latent_experiment/val')
parser.add_argument('--output_folder', type=str,default='latent_experiment/ex1/output')
parser.add_argument('--ckpt_path', type=str,default='ckpt-zinc9M/model-epoch=24-val_loss=3.40.ckpt')
parser.add_argument('--config', type=str, default='config.yaml')
parser.add_argument('--batch_size', type=int, default=100)
parser.add_argument('--device', type=str, default='cpu')

args, unknown = parser.parse_known_args()

In [8]:
import torch.nn as nn

# Extract encoder configuration from the config file
cfg = load_config(args.config)

# Create an encoder-only object
encoder = Encoder(**cfg['encoder_config'])
Wh_mu = nn.Linear(
    cfg['encoder_config']['hidden_dim'],
    cfg['optimal_layer_config']['latent_dim']
)
Wh_log_var = nn.Linear(cfg['encoder_config']['hidden_dim'], cfg['optimal_layer_config']['latent_dim'])
Wx_log_var = nn.Linear(cfg['encoder_config']['hidden_dim'], 1) 

encoder.load_state_dict(torch.load("weights/encoder_weights.pth"))
Wh_mu.load_state_dict(torch.load("weights/encoder_weights_KL.pth"))
Wh_log_var.load_state_dict(torch.load("weights/encoder_weights_KL_Wh_log_var.pth"))
Wx_log_var.load_state_dict(torch.load("weights/encoder_weights_KL_Wx_log_var.pth"))

# Include the KL divergence layers into the encoder model
encoder.Wh_mu = Wh_mu
encoder.Wh_log_var = Wh_log_var
encoder.Wx_log_var = Wx_log_var

encoder.eval()
print(encoder)

Encoder:
UniTransformerO2(num_blocks=1, num_layers=9, n_heads=16, act_fn=relu, norm=True, cutoff_mode=global, ew_net_type=r, init h emb: AttentionLayerO2TwoUpdateNodeGeneral(
  (distance_expansion): GaussianSmearing(start=0.0, stop=10.0, num_gaussians=20)
  (x2h_layers): ModuleList(
    (0): BaseX2HAttLayer(
      (hk_func): MLP(
        (net): Sequential(
          (0): Linear(in_features=340, out_features=128, bias=True)
          (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (2): ReLU()
          (3): Linear(in_features=128, out_features=128, bias=True)
        )
      )
      (hv_func): MLP(
        (net): Sequential(
          (0): Linear(in_features=340, out_features=128, bias=True)
          (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (2): ReLU()
          (3): Linear(in_features=128, out_features=128, bias=True)
        )
      )
      (hq_func): MLP(
        (net): Sequential(
          (0): Linear(in_features=128, out_features=1

In [9]:
# Load the molecules
mols=process_sdf_files_to_list(args.sdf_folder)

# Build a numbered atom batch
all_h = torch.cat([entry['h'] for entry in mols], dim=0)
all_x = torch.cat([entry['x'] for entry in mols], dim=0)
all_batch = []
current_index = 0
for entry in mols:
    atom_num = entry['atom_num']
    all_batch += [current_index] * atom_num
    current_index += 1
all_batch = torch.tensor(all_batch, dtype=torch.long)

Processing molecules:   0%|          | 0/932 [00:00<?, ?mol/s]

Processing molecules: 100%|██████████| 932/932 [00:00<00:00, 2116.01mol/s]


In [10]:
from torch_scatter import scatter_mean

def center_pos(ligand_pos,batch_ligand, mode=True):
    if mode == False:
        offset = 0.0
        pass
    elif mode == True:
        offset = scatter_mean(ligand_pos, batch_ligand, dim=0)
        ligand_pos = ligand_pos - offset[batch_ligand]
    else:
        raise NotImplementedError
    return ligand_pos, offset

MAP_ATOM_TYPE_ONLY_TO_INDEX = {
    6: 0,
    7: 1,
    8: 2,
    9: 3,
    15: 4,
    16: 5,
    17: 6,
    35: 7,
    53: 8,
}
MAP_INDEX_TO_ATOM_TYPE_ONLY = {v: k for k, v in MAP_ATOM_TYPE_ONLY_TO_INDEX.items()}


In [11]:
import torch
import torch.nn.functional as F

def molecule_to_latent(encoder, mol_entry, return_numpy=False):
    """
    Compute latent representation (Zh, Zx) for a single molecule entry from mol_1000.

    Args:
        encoder: the trained encoder component of the model with an `encode(one_hot_h, x, batch_ligand, deterministic=True)` method.
               Model should be in eval() mode (this function will not call model.eval() automatically).
               The function will detect device from model parameters (next(model.parameters()).device).
        mol_entry: dict-like containing at least:
                   - 'h': tensor of atom types (integers or torch tensors)
                   - 'x': tensor of shape (N, 3) with coordinates
                   - 'atom_num': integer (optional)
        return_numpy: if True, returns numpy arrays (Zh_cpu, Zx_cpu, global_batch_cpu). Default False.

    Returns:
        Zh, Zx, global_batch  (either torch tensors on model device or numpy arrays if return_numpy=True)
    """
    # load the config   
    cfg = load_config("config.yaml")

    # detect device from model (fallback to cpu)
    try:
        device = next(encoder.parameters()).device
    except StopIteration:
        device = torch.device("cpu")

    # pull data
    h = mol_entry['h']  # expect 1D tensor-like of atom types 
    x = mol_entry['x']  # expect shape (N, 3)

    # Ensure tensors and move to device
    x = x.to(device)
    # Convert h to torch.LongTensor on device. If h is already tensor, make sure it's long.
    if not isinstance(h, torch.Tensor):
        h = torch.tensor(h, dtype=torch.long, device=device)
    else:
        h = h.to(device).long()

    # Chech if atom types should be mapped to indices
    needs_mapping = False
    K = cfg['encoder_config']['ligand_v_dim']
    if h.max().item() >= K:  # likely not indices
        needs_mapping = True

    if needs_mapping:
        h = torch.tensor([MAP_ATOM_TYPE_ONLY_TO_INDEX[int(i.item())] for i in h], dtype=torch.long, device=device)

    # One-hot encode using encoder ligand dim K
    one_hot_h = F.one_hot(h, num_classes=K).float().to(device)

    # Build a batch vector for a single molecule: all zeros
    batch_ligand = torch.zeros_like(h, dtype=torch.long, device=device)

    # Center positions
    x_centered, _ = center_pos(x, batch_ligand, mode=True)

    # Encode into latent space
    model_device_before = device
    encoder.eval()
    with torch.no_grad():
        Zh, Zx, global_batch, Zh_kl_loss, Zx_kl_loss = encoder.encode(one_hot_h, x_centered, batch_ligand, deterministic=True)
        
    if return_numpy:
        Zh_cpu = Zh.cpu().numpy()
        Zx_cpu = Zx.cpu().numpy()
        global_batch_cpu = global_batch.cpu().numpy()
        return Zh_cpu, Zx_cpu, global_batch_cpu

    return Zh, Zx, global_batch


In [12]:
encoder.eval()
mol0 = mols[0]             # single molecule entry
Zh, Zx, global_batch = molecule_to_latent(encoder, mol0)
print("Zh shape:", Zh.shape)
print("Zx shape:", Zx.shape)


Zh shape: torch.Size([10, 32])
Zx shape: torch.Size([10, 3])


To do: run a file from the CCDC through the encoder

Also - some primary thoughts on the decoder